In [ ]:
#  test torchkbnufft with cardiac dataset : single phase multi coil reconstruction
#  cuda OOM(outofmemory) : multi phase multi coil reconstruction

In [ ]:
import numpy as np
import torch
import scipy.io as sio

!pip install torchkbnufft

import torchkbnufft as tkbn
import matplotlib.pyplot as plt

# Force CPU and float32 defaults
device = torch.device("cpu")
torch.set_default_dtype(torch.float32)

print("Using device:", device)
print("Torch:", torch.__version__)
print("NumPy:", np.__version__)

Using device: cpu
Torch: 2.11.0+cpu
NumPy: 2.0.2


In [ ]:
slices = 100
grid_size = (100, slices, 100)
k_size = 100
im_size = (k_size, slices, k_size)

# Create the adjoint NUFFT ON CPU
# adjnufft_ob = tkbn.KbNufftAdjoint(
#     im_size=im_size,
#     grid_size=grid_size,
# ).to(device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True) # Remount Drive to ensure it's fresh

Mounted at /content/drive


In [ ]:
mat_contents = sio.loadmat('/content/drive/MyDrive/ACR_data.mat') # ***Adjust this path if your file is in a different location in MyDrive***
data = mat_contents['a']  # adjust key if different
print("data shape:", data.shape)

kdatax = data[:,0].astype(np.float32)
kdatay = data[:,1].astype(np.float32)

ktrajx = data[:,2].astype(np.float32)
ktrajy = data[:,3].astype(np.float32)
ktrajz = data[:,4].astype(np.float32)

# kbins = data[:,5]

#
totalPoints = data.shape[0]

points = 510
shots = 26*32

data shape: (425984, 5)


In [ ]:
slices = 100
grid_size = (100, slices, 100)
k_size = 100
im_size = (k_size, slices, k_size)

#
adjnufft_ob = tkbn.KbNufftAdjoint(
    im_size=im_size,
    grid_size=grid_size,
).to(device)

ktrajs = []
kdatas = []
kcomps = []

images = []

# for fr in range(frames):

# print('frame=', fr)
# bin_frame = [(bin_pairs[i][0]+fr*frame_lens[i], bin_pairs[i][0]+frame_lens[i]*(fr+1)) for i in range(bins_len)]

kdatax = data[:,0].astype(np.float32)
kdatay = data[:,1].astype(np.float32)

ktrajx = data[:,2].astype(np.float32)
ktrajy = data[:,3].astype(np.float32)
ktrajz = data[:,4].astype(np.float32)

#    ktrajx_ranges = [ktrajx[start:end] for start, end in bin_frame]
#    ktrajy_ranges = [ktrajy[start:end] for start, end in bin_frame]
#    ktrajz_ranges = [ktrajz[start:end] for start, end in bin_frame]

#    ktrajx_bins = np.concatenate(ktrajx_ranges)
#    ktrajy_bins = np.concatenate(ktrajy_ranges)
#    ktrajz_bins = np.concatenate(ktrajz_ranges)

#
#    kdatax_ranges = [kdatax[start:end] for start, end in bin_frame]
#    kdatay_ranges = [kdatay[start:end] for start, end in bin_frame]

#    kdatax_bins = np.concatenate(kdatax_ranges)
#    kdatay_bins = np.concatenate(kdatay_ranges)

bin0_len = kdatax.shape[0]

#
dkx = ktrajx
dky = ktrajy
dkz = ktrajz
dk_mag = (dkx**2+dky**2+dkz**2)**0.5

#
ktraj = np.zeros((3, bin0_len))
ktraj[0,:] = dkx/dk_mag.max()*np.pi
ktraj[1,:] = dky/dk_mag.max()*np.pi
ktraj[2,:] = dkz/dk_mag.max()*np.pi

# convert k-space trajectory to a tensor
ktraj = torch.tensor(ktraj).to(device)
print('ktraj shape: {}'.format(ktraj.shape))

#
# density compensation
dcomp = tkbn.calc_density_compensation_function(ktraj=ktraj, im_size=im_size)

rawdata = kdatax + 1j*kdatay
kdata = torch.tensor(rawdata).to(device)

# kdata = torch.reshape(rawdata1, (channels, shots*readouts)).unsqueeze(0)
kdata = kdata.to(torch.complex128)

image_sharp = adjnufft_ob(kdata * dcomp, ktraj)

image = abs(image_sharp).cpu().squeeze()
print(image.shape)

images.append(image)

ktrajs.append(ktraj.cpu().squeeze().numpy())
kdatas.append(kdata.cpu().squeeze().numpy())
kcomps.append(dcomp.cpu().squeeze().numpy())

ktraj shape: torch.Size([3, 425984])
torch.Size([100, 100, 100])


In [ ]:
sio.savemat('ACR_test.mat', {'ktrajs':ktrajs, 'kdatas':kdatas, 'kcomps':kcomps})

In [ ]:
!ls /content/

ACR_test.mat  drive  sample_data


In [ ]:
import shutil
import os

# Define the source and destination paths
source_path = '/content/ACR_test.mat'
destination_folder = '/content/drive/MyDrive/' # Moves to the root of your Google Drive
destination_path = os.path.join(destination_folder, 'ACR_test.mat')

# Check if the source file exists
if os.path.exists(source_path):
    # Check if the destination folder exists, create if not
    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    shutil.move(source_path, destination_path)
    print(f"'ACR_test.mat' moved to Google Drive at: {destination_path}")
else:
    print(f"File not found at {source_path}. Please ensure it was created correctly.")

'ACR_test.mat' moved to Google Drive at: /content/drive/MyDrive/ACR_test.mat


After running the above cell, you should find `ACR_test.mat` in the root of your Google Drive (`MyDrive` folder) when you access it through the Google Drive interface (drive.google.com).

### Option 2: Download to Your Laptop

If you prefer to download `ACR_test.mat` directly to your local machine, you can use the `files.download` function from `google.colab`.

In [ ]:
from google.colab import files

file_to_download = '/content/ACR_test.mat'

if os.path.exists(file_to_download):
    files.download(file_to_download)
    print(f"Downloading '{file_to_download}' to your local machine.")
else:
    print(f"File not found at {file_to_download}. Please ensure it was created correctly.")

File not found at /content/ACR_test.mat. Please ensure it was created correctly.
